In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
%run ../0-common/env-config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [0]:
dim_constructors_df = (
    spark.table(f"{catalog_name}.{silver_schema}.constructors")
    .filter(F.col("batch_id") == v_batch_id)
    .select("constructor_id", "constructor_name", "nationality")
)

In [0]:
dim_constructors_df = (
    dim_constructors_df
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
    )

In [0]:
if not spark.catalog.tableExists(target_table):
    (
        dim_constructors_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )
else:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, target_table)
    (
        delta_table.alias("t")
        .merge(
            dim_constructors_df.alias("c"),
            "t.constructor_id = c.constructor_id"
        )
        .whenMatchedUpdate(
            set={
                "constructor_name": "c.constructor_name",
                "nationality": "c.nationality",
                "updated_at": "c.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )